# Modelo XGBoost — Fase 6, paso 1: baseline

Primer modelo XGBoost con hiperparámetros por defecto, para tener un punto de partida antes de ajustar nada. Se entrena con los datos que dejó listos `01_feature_engineering.ipynb` en `data/processed/xgboost/` (train real, sin SMOTE, y val real) y se evalúa **solo en val**.

Usa 55 features. `Destination Port`, `Source Port` y `Protocol` los quita el pipeline compartido (`src/preprocessing/clean.py`), no `01_feature_engineering.ipynb`. Salieron a partir de la auditoría del primer baseline (59 features): el puerto de destino separaba las clases por cómo se armó el laboratorio de CIC-IDS2017 y no por el comportamiento del ataque. El detalle está en `docs/auditoria-baseline-xgboost.md`.

`X_test`/`y_test` no se cargan en este notebook. Test se reserva para la evaluación final, cuando el modelo ya esté cerrado; si se usara para ajustar el umbral o comparar variantes, las métricas finales quedarían optimistas.

Este notebook es el baseline previo al tratamiento del desbalance: sin SMOTE y sin pesos de clase, que es la variante B de `02b_ablacion_smote.ipynb` y la referencia de esa ablación. Con el dataset original, la regla de decisión de `02b` eligió esta misma variante B; con el dataset corregido (443,614 filas) eligió la C, que entrena con el train real y `scale_pos_weight = benignos / ataques` de `y_train`. Desde `03_tuning.ipynb` en adelante se entrena con la variante C. La versión con SMOTE que genera el 01 se conserva solo para esa ablación.

Las métricas salen de `evaluate_model` en `src/evaluation/metrics.py`, la misma función que usan GRU y SVM.

In [1]:
import sys
from pathlib import Path

# Raíz del repo = primera carpeta hacia arriba que contenga src/.
# Así el notebook funciona sin importar desde dónde se abra el kernel.
RAIZ_REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(RAIZ_REPO) not in sys.path:
    sys.path.append(str(RAIZ_REPO))

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from src.evaluation.metrics import evaluate_model, Timer

DATA_DIR = RAIZ_REPO / "data" / "processed" / "xgboost"
print(f"Raíz del repo: {RAIZ_REPO}")
print(f"Datos:         {DATA_DIR}")
print(f"xgboost {xgb.__version__}")

Raíz del repo: C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids
Datos:         C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\data\processed\xgboost
xgboost 3.3.0


### Carga de train y val

Solo train (real, sin SMOTE) y val. `y` se lee como array 1D (la columna `label_binaria`).

In [2]:
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")["label_binaria"].to_numpy()
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
y_val = pd.read_csv(DATA_DIR / "y_val.csv")["label_binaria"].to_numpy()

assert list(X_train.columns) == list(X_val.columns), "Columnas distintas entre train y val"

# Guarda: X_train.csv debe ser el train real. Si trae filas sintéticas, el 01 dejó la versión con SMOTE.
N_TRAIN_ESPERADO = 310_529
N_ATAQUES_TRAIN_ESPERADO = 8_230
assert len(X_train) == N_TRAIN_ESPERADO and y_train.sum() == N_ATAQUES_TRAIN_ESPERADO, (
    f"Se esperaban {N_TRAIN_ESPERADO:,} filas y {N_ATAQUES_TRAIN_ESPERADO:,} ataques en el train real, "
    f"hay {len(X_train):,} y {y_train.sum():,}. Revisar que 01_feature_engineering.ipynb guarde el train sin SMOTE."
)

for nombre, X, y in [("train", X_train, y_train), ("val", X_val, y_val)]:
    print(f"{nombre:>5}: X {X.shape} | y {y.shape} | ataques: {y.sum()} ({y.mean():.4%})")

train: X (310529, 55) | y (310529,) | ataques: 8230 (2.6503%)
  val: X (66541, 55) | y (66541,) | ataques: 1884 (2.8313%)


### Dispositivo: GPU (CUDA) si está disponible

No alcanza con que xgboost esté compilado con CUDA; también tiene que haber una GPU usable. Por eso se hace un fit de prueba chiquito con `device="cuda"` y, si falla, se usa CPU.

In [3]:
def gpu_disponible_para_xgboost() -> bool:
    if not xgb.build_info().get("USE_CUDA", False):
        return False
    try:
        rng = np.random.default_rng(0)
        X_prueba = rng.normal(size=(200, 5))
        y_prueba = (X_prueba[:, 0] > 0).astype(int)
        xgb.XGBClassifier(n_estimators=2, tree_method="hist", device="cuda").fit(X_prueba, y_prueba)
        return True
    except xgb.core.XGBoostError:
        return False


DEVICE = "cuda" if gpu_disponible_para_xgboost() else "cpu"
print(f"USE_CUDA en el build: {xgb.build_info().get('USE_CUDA')}")
print(f"Dispositivo para entrenar: {DEVICE}")

USE_CUDA en el build: True
Dispositivo para entrenar: cuda


### Modelo baseline

Hiperparámetros por defecto de XGBoost salvo:

- `n_estimators=1000` con `early_stopping_rounds=50`: deja que el propio val decida cuántos árboles usar, en vez de fijar un número a ciegas.
- `eval_metric="aucpr"`: con una proporción tan baja de ataques en val, el área bajo la curva precision-recall es más informativa que ROC-AUC o logloss para decidir cuándo parar.
- Sin SMOTE y sin `scale_pos_weight`: es la variante B de `02b_ablacion_smote.ipynb`, el punto de partida contra el que se comparan las formas de compensar el desbalance. La regla de `02b` eligió B con el dataset original y C (`scale_pos_weight`) con el dataset corregido de 443,614 filas.

In [4]:
modelo = xgb.XGBClassifier(
    n_estimators=1000,
    early_stopping_rounds=50,
    eval_metric="aucpr",
    tree_method="hist",
    device=DEVICE,
    random_state=42,
    n_jobs=-1,
)

with Timer() as t:
    modelo.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

print(f"\nDispositivo: {DEVICE} | tiempo de entrenamiento: {t.elapsed:.2f}s")
print(f"best_iteration: {modelo.best_iteration} | mejor aucpr en val: {modelo.best_score:.6f}")

[0]	validation_0-aucpr:0.91702


[100]	validation_0-aucpr:0.99432


[105]	validation_0-aucpr:0.99424



Dispositivo: cuda | tiempo de entrenamiento: 0.85s
best_iteration: 55 | mejor aucpr en val: 0.994426


### Evaluación en val con umbral 0.5

`save_dir=None` para que no escriba `reports/metrics/`: esto es un baseline en val, no el resultado final que entra a la comparación de la Fase 8.

Antes de predecir se pasa el modelo a CPU (`set_params(device="cpu")`). `X_val` es un DataFrame en memoria de CPU, y si el booster sigue en cuda XGBoost avisa que los dispositivos no coinciden y predice por un camino más lento.

In [5]:
modelo.set_params(device="cpu")

y_proba_val = modelo.predict_proba(X_val)[:, 1]
y_pred_val = (y_proba_val >= 0.5).astype(int)

resultados_val = evaluate_model(
    y_val,
    y_pred_val,
    y_proba_val,
    model_name="xgboost_baseline_val",
    training_time_seconds=t.elapsed,
    save_dir=None,
)

pr_auc_val = average_precision_score(y_val, y_proba_val)
print(f"\nPR-AUC (average precision) en val: {pr_auc_val:.6f}")
resultados_val


=== Resultados: xgboost_baseline_val ===
Accuracy:  0.9975
Precision: 0.9920
Recall:    0.9183  <- métrica principal
F1-score:  0.9537
ROC-AUC:   0.9999
Matriz de confusión -> TN: 64643, FP: 14, FN: 154, TP: 1730
Tiempo de entrenamiento: 0.85s

PR-AUC (average precision) en val: 0.994515


{'modelo': 'xgboost_baseline_val',
 'accuracy': 0.9974752408289626,
 'precision': 0.9919724770642202,
 'recall': 0.9182590233545648,
 'f1_score': 0.9536934950385888,
 'matriz_confusion': {'verdaderos_negativos': 64643,
  'falsos_positivos': 14,
  'falsos_negativos': 154,
  'verdaderos_positivos': 1730},
 'tasa_falsos_negativos': np.float64(0.08174097664543524),
 'tiempo_entrenamiento_segundos': 0.8523330688476562,
 'roc_auc': 0.9999168731211282}

### Barrido de umbrales en val

0.5 no tiene nada de especial: el modelo se entrena con la proporción real de ataques, sin SMOTE ni pesos, así que sus probabilidades reflejan esa proporción y el mejor umbral puede quedar lejos de 0.5 en cualquiera de los dos sentidos (en `02b` se vio que SMOTE era lo que corría la calibración). La tabla muestra cómo se mueven precision, recall y F1 al cambiar el umbral en pasos de 0.05. Después se hace una búsqueda fina, de 0.01 a 0.99 en pasos de 0.01. Acá solo se reporta; la decisión del umbral se toma aparte.

In [6]:
umbrales = np.round(np.arange(0.05, 0.951, 0.05), 2)

filas = []
for u in umbrales:
    pred = (y_proba_val >= u).astype(int)
    filas.append({
        "umbral": u,
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
        "predichos_ataque": int(pred.sum()),
    })
tabla_umbrales = pd.DataFrame(filas)

with pd.option_context("display.float_format", "{:.4f}".format):
    print(tabla_umbrales.to_string(index=False))

 umbral  precision  recall     f1  predichos_ataque
 0.0500     0.9920  0.9199 0.9546              1747
 0.1000     0.9920  0.9193 0.9543              1746
 0.1500     0.9920  0.9183 0.9537              1744
 0.2000     0.9920  0.9183 0.9537              1744
 0.2500     0.9920  0.9183 0.9537              1744
 0.3000     0.9920  0.9183 0.9537              1744
 0.3500     0.9920  0.9183 0.9537              1744
 0.4000     0.9920  0.9183 0.9537              1744
 0.4500     0.9920  0.9183 0.9537              1744
 0.5000     0.9920  0.9183 0.9537              1744
 0.5500     0.9920  0.9183 0.9537              1744
 0.6000     0.9920  0.9183 0.9537              1744
 0.6500     0.9920  0.9183 0.9537              1744
 0.7000     0.9920  0.9183 0.9537              1744
 0.7500     0.9920  0.9183 0.9537              1744
 0.8000     0.9920  0.9183 0.9537              1744
 0.8500     0.9920  0.9167 0.9528              1741
 0.9000     0.9931  0.9114 0.9505              1729
 0.9500     

In [7]:
RECALL_MINIMO = 0.75

candidatos = tabla_umbrales[tabla_umbrales["recall"] >= RECALL_MINIMO]
if candidatos.empty:
    umbral_optimo = None
    print(f"Ningún umbral de la tabla alcanza recall >= {RECALL_MINIMO} en val.")
else:
    umbral_optimo = candidatos.loc[candidatos["f1"].idxmax()]
    print(f"Umbral que maximiza F1 con recall >= {RECALL_MINIMO} (entre los de la tabla):")
    print(f"  umbral:    {umbral_optimo['umbral']:.2f}")
    print(f"  precision: {umbral_optimo['precision']:.4f}")
    print(f"  recall:    {umbral_optimo['recall']:.4f}")
    print(f"  f1:        {umbral_optimo['f1']:.4f}")

Umbral que maximiza F1 con recall >= 0.75 (entre los de la tabla):
  umbral:    0.05
  precision: 0.9920
  recall:    0.9199
  f1:        0.9546


In [8]:
# Búsqueda fina: mismo criterio que arriba, en pasos de 0.01.
# Si varios umbrales empatan en F1, idxmax se queda con el más bajo (el de más recall).
umbrales_finos = np.round(np.arange(0.01, 0.991, 0.01), 2)

filas_finas = []
for u in umbrales_finos:
    pred = (y_proba_val >= u).astype(int)
    filas_finas.append({
        "umbral": u,
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
    })
tabla_umbrales_fina = pd.DataFrame(filas_finas)

candidatos_finos = tabla_umbrales_fina[tabla_umbrales_fina["recall"] >= RECALL_MINIMO]
if candidatos_finos.empty:
    umbral_optimo_fino = None
    print(f"Ningún umbral de la búsqueda fina alcanza recall >= {RECALL_MINIMO} en val.")
else:
    umbral_optimo_fino = candidatos_finos.loc[candidatos_finos["f1"].idxmax()]
    tn, fp, fn, tp = confusion_matrix(
        y_val, (y_proba_val >= umbral_optimo_fino["umbral"]).astype(int)
    ).ravel()
    print(f"Umbral que maximiza F1 con recall >= {RECALL_MINIMO} (búsqueda fina, pasos de 0.01):")
    print(f"  umbral:    {umbral_optimo_fino['umbral']:.2f}")
    print(f"  precision: {umbral_optimo_fino['precision']:.4f}")
    print(f"  recall:    {umbral_optimo_fino['recall']:.4f}")
    print(f"  f1:        {umbral_optimo_fino['f1']:.4f}")
    print(f"  TN: {tn} | FP: {fp} | FN: {fn} | TP: {tp}")

Umbral que maximiza F1 con recall >= 0.75 (búsqueda fina, pasos de 0.01):
  umbral:    0.01
  precision: 0.9862
  recall:    0.9830
  f1:        0.9846
  TN: 64631 | FP: 26 | FN: 32 | TP: 1852


### Importancia de features (gain)

`gain` = cuánto mejora en promedio la función de pérdida cada vez que un árbol parte por esa feature. Las features que el modelo nunca usó no aparecen en `get_score`, por eso se completan con 0.

Se señala aparte `Init_Win_bytes_backward`, la ventana TCP inicial que anuncia el servidor, que depende de cómo está configurado ese host y no del ataque. Si pesa mucho, puede que el modelo esté aprendiendo detalles de esta captura en vez del patrón del ataque, igual que pasaba con los puertos.

In [9]:
gain = modelo.get_booster().get_score(importance_type="gain")
importancias = (
    pd.DataFrame({"feature": X_train.columns})
    .assign(gain=lambda d: d["feature"].map(gain).fillna(0.0))
    .sort_values("gain", ascending=False, ignore_index=True)
)
importancias["gain_relativo"] = importancias["gain"] / importancias["gain"].sum()
importancias.insert(0, "posicion", importancias.index + 1)

print("Top 15 features por gain:")
with pd.option_context("display.float_format", "{:.4f}".format):
    print(importancias.head(15).to_string(index=False))

print("\nFeatures a vigilar (posible dependencia de la captura):")
with pd.option_context("display.float_format", "{:.4f}".format):
    print(importancias[importancias["feature"].isin(
        ["Protocol", "Init_Win_bytes_backward"]
    )].to_string(index=False))

print(f"\nFeatures con gain 0 (nunca usadas): {(importancias['gain'] == 0).sum()} de {len(importancias)}")

Top 15 features por gain:
 posicion                     feature       gain  gain_relativo
        1                  Active Min 12985.7119         0.2911
        2        min_seg_size_forward  7263.9976         0.1629
        3                 Bwd IAT Max  6303.9814         0.1413
        4                    Idle Std  4213.7920         0.0945
        5      Bwd Packet Length Mean  3143.5193         0.0705
        6 Total Length of Bwd Packets  2228.1040         0.0500
        7         Average Packet Size  1165.3712         0.0261
        8           Max Packet Length  1039.2694         0.0233
        9       Bwd Packet Length Std   786.9901         0.0176
       10                   Idle Mean   699.8009         0.0157
       11              PSH Flag Count   690.9845         0.0155
       12 Total Length of Fwd Packets   608.7323         0.0136
       13       Fwd Packet Length Std   585.0495         0.0131
       14       Fwd Packet Length Min   571.1204         0.0128
       15     

### Verificaciones: overfitting y fuga

Dos pruebas para descartar que el resultado en val venga de memorizar train o de información que se filtra entre splits.

**Overfitting (train vs val).** Se evalúa el modelo sobre train (real, sin SMOTE) y se compara con val, las dos con umbral 0.5. Si en train el modelo es mucho mejor que en val, está memorizando.

**Etiquetas mezcladas (fuga).** Se mezcla `y_train` al azar y se entrena el mismo modelo, con val y sus etiquetas reales como `eval_set`. Al mezclar, se rompe la relación entre features y etiqueta, así que un modelo sin fuga debería quedar al nivel del azar en val: ROC-AUC cerca de 0.5 y PR-AUC cerca de la proporción de ataques en val. Si con etiquetas mezcladas igual detecta bien los ataques de val, la información está llegando por otro lado (por ejemplo, filas repetidas entre train y val). El early stopping elige la iteración mirando el mismo val, así que con 0 o 1 árboles puede quedar algo por encima del azar por suerte.

In [10]:
from sklearn.metrics import roc_auc_score


def metricas_umbral(y, proba, umbral=0.5):
    pred = (proba >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "recall": recall_score(y, pred, zero_division=0),
        "precision": precision_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "roc_auc": roc_auc_score(y, proba),
        "pr_auc": average_precision_score(y, proba),
        "FP": int(fp),
        "FN": int(fn),
    }


metricas_train = metricas_umbral(y_train, modelo.predict_proba(X_train)[:, 1])
metricas_val = metricas_umbral(y_val, y_proba_val)
tabla_overfitting = pd.DataFrame({
    f"train ({len(y_train):,})": metricas_train,
    f"val ({len(y_val):,})": metricas_val,
})
with pd.option_context("display.float_format", "{:.6f}".format):
    print("Overfitting — umbral 0.5:")
    print(tabla_overfitting.to_string())

Overfitting — umbral 0.5:
           train (310,529)  val (66,541)
recall            0.999635      0.918259
precision         0.991085      0.991972
f1                0.995342      0.953693
roc_auc           0.999976      0.999917
pr_auc            0.998541      0.994515
FP               74.000000     14.000000
FN                3.000000    154.000000


In [11]:
# Mismos parámetros que el baseline, pero entrenando en DEVICE (el baseline quedó en cpu
# para predecir). Cada semilla usa un modelo nuevo; `modelo` no se toca.
params_baseline = {**modelo.get_params(), "device": DEVICE}

filas_mezcla = []
for semilla in [0, 1, 2]:
    y_train_mezclado = np.random.default_rng(semilla).permutation(y_train)
    modelo_mezcla = xgb.XGBClassifier(**params_baseline)
    modelo_mezcla.fit(X_train, y_train_mezclado, eval_set=[(X_val, y_val)], verbose=False)
    modelo_mezcla.set_params(device="cpu")
    m = metricas_umbral(y_val, modelo_mezcla.predict_proba(X_val)[:, 1])
    filas_mezcla.append({
        "semilla": semilla,
        "roc_auc_val": m["roc_auc"],
        "pr_auc_val": m["pr_auc"],
        "best_iteration": modelo_mezcla.best_iteration,
        "recall@0.5": m["recall"],
        "FP@0.5": m["FP"],
    })
tabla_mezcla = pd.DataFrame(filas_mezcla)

print(f"Referencia: proporción de ataques en val = {y_val.mean():.6f} (PR-AUC de un modelo al azar)\n")
with pd.option_context("display.float_format", "{:.6f}".format):
    print("Etiquetas de train mezcladas — métricas en val con etiquetas reales:")
    print(tabla_mezcla.to_string(index=False))

Referencia: proporción de ataques en val = 0.028313 (PR-AUC de un modelo al azar)

Etiquetas de train mezcladas — métricas en val con etiquetas reales:
 semilla  roc_auc_val  pr_auc_val  best_iteration  recall@0.5  FP@0.5
       0     0.478556    0.050934               5    0.000000       0
       1     0.540013    0.089753               5    0.000000       0
       2     0.438399    0.076668              21    0.000000       0


### Resumen

In [12]:
print("=" * 60)
print("RESUMEN — Fase 6, paso 1: baseline XGBoost (solo val)")
print("=" * 60)
print(f"Dispositivo:              {DEVICE}")
print(f"Tiempo de entrenamiento:  {t.elapsed:.2f}s")
print(f"best_iteration:           {modelo.best_iteration}")
print(f"PR-AUC val:               {pr_auc_val:.6f}")
print(f"ROC-AUC val:              {resultados_val['roc_auc']:.6f}")
print(f"Umbral 0.5 -> precision {resultados_val['precision']:.4f} | "
      f"recall {resultados_val['recall']:.4f} | F1 {resultados_val['f1_score']:.4f}")
if umbral_optimo is not None:
    print(f"Umbral {umbral_optimo['umbral']:.2f} (máx F1, recall >= {RECALL_MINIMO}) -> "
          f"precision {umbral_optimo['precision']:.4f} | recall {umbral_optimo['recall']:.4f} | "
          f"F1 {umbral_optimo['f1']:.4f}")
if umbral_optimo_fino is not None:
    print(f"Umbral {umbral_optimo_fino['umbral']:.2f} (búsqueda fina) -> "
          f"precision {umbral_optimo_fino['precision']:.4f} | recall {umbral_optimo_fino['recall']:.4f} | "
          f"F1 {umbral_optimo_fino['f1']:.4f}")
print(f"Overfitting (umbral 0.5) -> recall train {metricas_train['recall']:.4f} vs "
      f"val {metricas_val['recall']:.4f} | FP {metricas_train['FP']} vs {metricas_val['FP']} | "
      f"FN {metricas_train['FN']} vs {metricas_val['FN']}")
print(f"Etiquetas mezcladas (semillas 0-2) -> ROC-AUC val {tabla_mezcla['roc_auc_val'].min():.4f}"
      f"-{tabla_mezcla['roc_auc_val'].max():.4f} | PR-AUC val {tabla_mezcla['pr_auc_val'].min():.4f}"
      f"-{tabla_mezcla['pr_auc_val'].max():.4f} (azar {y_val.mean():.4f}) | "
      f"recall@0.5 máx {tabla_mezcla['recall@0.5'].max():.4f}")

RESUMEN — Fase 6, paso 1: baseline XGBoost (solo val)
Dispositivo:              cuda
Tiempo de entrenamiento:  0.85s
best_iteration:           55
PR-AUC val:               0.994515
ROC-AUC val:              0.999917
Umbral 0.5 -> precision 0.9920 | recall 0.9183 | F1 0.9537
Umbral 0.05 (máx F1, recall >= 0.75) -> precision 0.9920 | recall 0.9199 | F1 0.9546
Umbral 0.01 (búsqueda fina) -> precision 0.9862 | recall 0.9830 | F1 0.9846
Overfitting (umbral 0.5) -> recall train 0.9996 vs val 0.9183 | FP 74 vs 14 | FN 3 vs 154
Etiquetas mezcladas (semillas 0-2) -> ROC-AUC val 0.4384-0.5400 | PR-AUC val 0.0509-0.0898 (azar 0.0283) | recall@0.5 máx 0.0000
